# 59 · The frontier: what changed in the last year
Level L6 · Part 7 · deep · ~30 min · builds on 44 (also uses 09, 40, 53, 55)

## Goal
Build a dated table of the last ~12 months' changes to the vendor APIs Scout uses, check each claim
against the SDKs installed on your laptop, list which lessons and which parts of Scout each one touches,
and set up a routine that reads vendor changelogs and turns them into a short to-do list.

## The idea
Everything in this course was checked in October 2026, and it will drift. In one year the vendors made
conversations stateful on the server, replaced sampling knobs with an **effort** dial, moved tools onto
their own machines, made structured output a standard feature, and swapped the HTTP library under two
SDKs. A **frontier table** keeps that history as data: date, vendor, change, source, lessons affected.
Each row gets a **probe**, a small offline check that the claim is still true in your venv.

Analogy: a ship's chart with dated corrections. You do not redraw the coast every week; you apply the
notices to mariners and note the date you did.

## Picture
```mermaid
flowchart LR
    C["vendor changelogs<br/>+ PyPI"] --> T["triage<br/>act / read / skip"]
    T --> F["frontier table<br/>dated rows"]
    F --> P["probe each row<br/>in your venv"]
    P --> L["lessons + Scout parts<br/>it touches"]
    L --> D["to-do list<br/>adopt / watch / no change"]
```

## Step by step
### 1. Setup: what Scout is built on today
Scout v2 is the package from lesson 55. Its settings live in one file, so we read them rather than
retype them. `AS_OF` is the day this table was checked: every date comparison uses it, so the lesson
prints the same thing next year (your own routine would use `date.today()`).

In [1]:
import inspect, json, re, sys, typing, urllib.request
from dataclasses import dataclass
from datetime import date, timedelta
from importlib.metadata import version
from pathlib import Path

import anthropic, openai
from google import genai
from google.genai import types

sys.path.insert(0, "../apps/55")
from scout.config import MODEL as SCOUT_MODEL

AS_OF = date(2026, 10, 11)
SCOUT = {"api": "Gemini generate_content on Vertex AI (express mode)", "model": SCOUT_MODEL["model"],
         "effort": f"thinking_level={SCOUT_MODEL['thinking_level']}",
         "tools": "own Python tools: search_notes, calculator, write_file",
         "tests": "vcrpy cassettes through llm_replay (lesson 47)"}
INSTALLED = {p: version(p) for p in ("anthropic", "openai", "google-genai", "httpx2", "vcrpy")}
for k, v in {**SCOUT, "installed": INSTALLED}.items():
    print(f"{k:10} {v}")

api        Gemini generate_content on Vertex AI (express mode)
model      gemini-3.5-flash-lite
effort     thinking_level=MINIMAL
tools      own Python tools: search_notes, calculator, write_file
tests      vcrpy cassettes through llm_replay (lesson 47)
installed  {'anthropic': '1.13.0', 'openai': '3.28.0', 'google-genai': '2.29.0', 'httpx2': '2.13.1', 'vcrpy': '8.3.0'}


### 2. The frontier table, as data
One row per change. `when` is the most precise date the source gives (a month if that is all it says).
`source` says where the date comes from: an SDK changelog entry is exact; a docs page often gives only
a month. `probe` names an offline check from step 3, or is empty when only the vendor's servers can
show it (a shut-down API looks the same as a live one from inside the SDK).

In [2]:
@dataclass
class Change:
    when: str             # "YYYY-MM-DD", or "YYYY-MM" when only the month is published
    vendor: str
    what: str
    source: str
    lessons: list[str]
    probe: str = ""

print("columns:", list(Change.__dataclass_fields__))

columns: ['when', 'vendor', 'what', 'source', 'lessons', 'probe']


The rows, from SDK changelogs, vendor docs and RESEARCH.md. The last Google row is copied from research
notes on purpose: step 3 checks it.

In [3]:
FRONTIER = [
    Change("2025-09-29", "Anthropic", "'Context engineering' essay: choose the smallest set of tokens per call",
           "anthropic.com engineering essay", ["31", "32"]),
    Change("2026-01-29", "Anthropic", "Structured outputs in the Messages API via output_config.format",
           "anthropic SDK 0.77.0 changelog; beta header dropped in 1.9.0 (2026-09-28)", ["20", "21"], "structured"),
    Change("2026-02-05", "Anthropic", "Adaptive thinking + effort replace budget_tokens (Opus 4.6; 400 on 4.7+)",
           "anthropic SDK 0.78.0 changelog; Opus 5.5 migration guide", ["04", "40"], "effort"),
    Change("2026-05-21", "Anthropic", "Newest dated server tool type: code_execution_20260521",
           "date inside the tool type name; tool reference", ["44"], "server_tools"),
    Change("2026-06", "Google", "Interactions API GA: server-side state, steps; generate_content legacy",
           "ai.google.dev interactions docs; in SDK since 1.55.0 (2025-12-11)", ["07", "09", "10"], "interactions"),
    Change("2026-08-12", "OpenAI", "openai 3.0.0: httpx2 is the default HTTP client",
           "openai SDK changelog", ["14", "47"], "httpx2"),
    Change("2026-08-20", "Anthropic", "anthropic 1.0.0: httpx2; Text Completions removed",
           "anthropic SDK changelog, MIGRATION.md", ["10", "47"], "httpx2"),
    Change("2026-08-20", "Google", "google-genai 2.x moved to httpx2 (claim to check)",
           "RESEARCH.md section 0, unverified", ["47"], "httpx2"),
    Change("2026-08-26", "OpenAI", "Assistants API shut down: use Responses + Conversations",
           "OpenAI migration guide", ["09", "10"]),
]
for c in sorted(FRONTIER, key=lambda c: c.when):
    print(f"{c.when:10} {c.vendor:9} {c.what[:62]:62} -> lessons {', '.join(c.lessons)}")

2025-09-29 Anthropic 'Context engineering' essay: choose the smallest set of tokens -> lessons 31, 32
2026-01-29 Anthropic Structured outputs in the Messages API via output_config.forma -> lessons 20, 21
2026-02-05 Anthropic Adaptive thinking + effort replace budget_tokens (Opus 4.6; 40 -> lessons 04, 40
2026-05-21 Anthropic Newest dated server tool type: code_execution_20260521         -> lessons 44
2026-06    Google    Interactions API GA: server-side state, steps; generate_conten -> lessons 07, 09, 10
2026-08-12 OpenAI    openai 3.0.0: httpx2 is the default HTTP client                -> lessons 14, 47
2026-08-20 Anthropic anthropic 1.0.0: httpx2; Text Completions removed              -> lessons 10, 47
2026-08-20 Google    google-genai 2.x moved to httpx2 (claim to check)              -> lessons 47
2026-08-26 OpenAI    Assistants API shut down: use Responses + Conversations        -> lessons 09, 10


### 3. Probe every claim in your venv
A probe asks the installed SDK a yes/no question without sending a request: does this method exist,
which values does this field accept, which HTTP library does the client sit on. The HTTP probe reads a
private attribute (`_client`): fine for a check, never for app code.

In [4]:
def http_lib(obj) -> str:
    """Which HTTP library a client object is built on: walk its class tree."""
    return next(c.__module__.split(".")[0] for c in type(obj).__mro__ if c.__module__.split(".")[0] in ("httpx", "httpx2"))

PROBES = {
    "httpx2": lambda: {"anthropic": http_lib(anthropic.Anthropic(api_key="probe")._client),
                       "openai": http_lib(openai.OpenAI(api_key="probe")._client),
                       "google-genai": http_lib(genai.Client(api_key="probe")._api_client._httpx_client)},
    "interactions": lambda: hasattr(genai.Client(api_key="probe"), "interactions"),
    "structured": lambda: "output_config" in inspect.signature(anthropic.Anthropic(api_key="probe").messages.create).parameters,
    "server_tools": lambda: sorted(n for n in dir(anthropic.types) if re.fullmatch(r"(WebSearch|CodeExecution)Tool\d{8}Param", n)),
}
for name, probe in PROBES.items():
    print(f"{name:13} {probe()}")

httpx2        {'anthropic': 'httpx2', 'openai': 'httpx2', 'google-genai': 'httpx'}


interactions  True
structured    True
server_tools  ['CodeExecutionTool20250522Param', 'CodeExecutionTool20250825Param', 'CodeExecutionTool20260120Param', 'CodeExecutionTool20260521Param', 'WebSearchTool20250305Param', 'WebSearchTool20260209Param', 'WebSearchTool20260318Param']


Three probes agree with the table. One does not: `google-genai` 2.28 still builds its client on
**httpx**, not httpx2. Its changelog says why: 2.18.0 (2026-08-12) added *support for injecting* an
httpx2 client; the default did not change. The row came from research notes, and the probe caught it.
This matters for Scout: tools that hook one HTTP library (recorders, tracers) see two of the three SDKs
through httpx2 and Gemini through httpx. Correct the row from the evidence.

In [5]:
row = next(c for c in FRONTIER if c.vendor == "Google" and c.probe == "httpx2")
row.when, row.what = "2026-08-12", "google-genai 2.18.0: an httpx2 client can be injected; default stays httpx"
row.source = "google-genai SDK changelog + probe on 2.28.0"
print(row)

Change(when='2026-08-12', vendor='Google', what='google-genai 2.18.0: an httpx2 client can be injected; default stays httpx', source='google-genai SDK changelog + probe on 2.28.0', lessons=['47'], probe='httpx2')


### 4. Effort controls: one dial, three spellings
**Effort controls** set how much a model thinks before it answers (lesson 40). Every vendor has one now,
with different names and different allowed values. Read the allowed values straight from the SDK types,
so the list updates when you upgrade.

In [6]:
def literals(t) -> list[str]:
    out = []
    for a in typing.get_args(t):
        out += literals(a) if typing.get_args(a) else ([a] if isinstance(a, str) else [])
    return out

from openai.types.shared import ReasoningEffort
EFFORT = {"gemini  thinking_level": [m.name.lower() for m in types.ThinkingLevel if m.name != "THINKING_LEVEL_UNSPECIFIED"],
          "anthropic output_config.effort": literals(typing.get_type_hints(anthropic.types.OutputConfigParam)["effort"]),
          "openai  reasoning.effort": literals(ReasoningEffort)}
for k, v in EFFORT.items():
    print(f"{k:31} {v}")
PROBES["effort"] = lambda: EFFORT

gemini  thinking_level          ['minimal', 'low', 'medium', 'high']
anthropic output_config.effort  ['low', 'medium', 'high', 'xhigh', 'max']
openai  reasoning.effort        ['none', 'minimal', 'low', 'medium', 'high', 'xhigh', 'max']


Scout runs at `MINIMAL`. Anthropic has no `minimal`. Porting Scout means picking the nearest level on
purpose, not letting a request fail. A small translator makes the choice visible.

In [7]:
def effort_for(vendor_key: str, level: str) -> str:
    allowed = EFFORT[vendor_key]
    if level in allowed:
        return level
    order = ["none", "minimal", "low", "medium", "high", "xhigh", "max"]
    above = [l for l in order[order.index(level):] if l in allowed]
    return above[0]                  # the cheapest level that still thinks at least as much

for key in EFFORT:
    print(f"{key:31} Scout's 'minimal' -> {effort_for(key, 'minimal')}")

gemini  thinking_level          Scout's 'minimal' -> minimal
anthropic output_config.effort  Scout's 'minimal' -> low
openai  reasoning.effort        Scout's 'minimal' -> minimal


### 5. Which lessons, and which parts of Scout, does each change touch?
Turn the table around: lesson -> changes. A lesson with rows next to it is the first place to re-read
after an upgrade.

In [8]:
by_lesson: dict[str, list[str]] = {}
for c in FRONTIER:
    for lesson in c.lessons:
        by_lesson.setdefault(lesson, []).append(f"{c.when} {c.vendor}")
for lesson in sorted(by_lesson):
    print(f"lesson {lesson}: {'; '.join(by_lesson[lesson])}")

lesson 04: 2026-02-05 Anthropic
lesson 07: 2026-06 Google
lesson 09: 2026-06 Google; 2026-08-26 OpenAI
lesson 10: 2026-06 Google; 2026-08-20 Anthropic; 2026-08-26 OpenAI
lesson 14: 2026-08-12 OpenAI
lesson 20: 2026-01-29 Anthropic
lesson 21: 2026-01-29 Anthropic
lesson 31: 2025-09-29 Anthropic
lesson 32: 2025-09-29 Anthropic
lesson 40: 2026-02-05 Anthropic
lesson 44: 2026-05-21 Anthropic
lesson 47: 2026-08-12 OpenAI; 2026-08-20 Anthropic; 2026-08-12 Google


Now the question this lesson set: which parts of Scout would you build differently today? Each answer is
**adopt now**, **watch** (re-check on a date) or **no change**, with the reason. Lesson 09 already showed
why the Interactions API is a "watch": our Vertex express-mode client got a 404 for it.

In [9]:
SCOUT_DECISIONS = [
    ("api",     "watch",     "Interactions API is Gemini Developer API only; Vertex gave 404 (lesson 09). Re-check quarterly."),
    ("effort",  "no change", "thinking_level=MINIMAL already is an effort control (lesson 40)."),
    ("tools",   "no change", "Notes are on the user's machine; a vendor-run tool cannot read them (lesson 44)."),
    ("tests",   "adopt now", "Assert in CI that cassettes still replay for each SDK's HTTP library (httpx vs httpx2)."),
    ("context", "no change", "Scout already sends only search hits, not whole notes (lessons 31-32)."),
    ("port",    "watch",     "On Claude, structured output is now GA: the pydantic retry loop (21) would rarely fire."),
]
for part, verdict, why in SCOUT_DECISIONS:
    print(f"{part:8} {verdict:10} {why}")

api      watch      Interactions API is Gemini Developer API only; Vertex gave 404 (lesson 09). Re-check quarterly.
effort   no change  thinking_level=MINIMAL already is an effort control (lesson 40).
tools    no change  Notes are on the user's machine; a vendor-run tool cannot read them (lesson 44).
tests    adopt now  Assert in CI that cassettes still replay for each SDK's HTTP library (httpx vs httpx2).
context  no change  Scout already sends only search hits, not whole notes (lessons 31-32).
port     watch      On Claude, structured output is now GA: the pydantic retry loop (21) would rarely fire.


### 6. A changelog routine: fetch
The table goes stale unless something feeds it. The routine: for each SDK on a **watch list**, read the
latest version from PyPI and the changelog entries newer than what you have installed. This cell downloads
them (tagged `needs-download`) and saves them to `../data/recorded/59.json`; the next cell reads that file.

In [10]:
WATCH = {"anthropic": "https://raw.githubusercontent.com/anthropics/anthropic-sdk-python/main/CHANGELOG.md",
         "openai": "https://raw.githubusercontent.com/openai/openai-python/main/CHANGELOG.md",
         "google-genai": "https://raw.githubusercontent.com/googleapis/python-genai/main/CHANGELOG.md"}
HEAD = re.compile(r"^## \[?(\d+)\.(\d+)\.(\d+)\]?.*?\((\d{4}-\d{2}-\d{2})\)", re.M)

def as_tuple(v: str) -> tuple[int, ...]:
    return tuple(int(x) for x in v.split(".")[:3])

def entries_since(text: str, installed: str) -> list[dict]:
    """Changelog entries newer than the installed version, newest first."""
    heads, out = list(HEAD.finditer(text)), []
    for h, nxt in zip(heads, heads[1:] + [None]):
        if as_tuple(".".join(h.groups()[:3])) <= as_tuple(installed):
            break
        out.append({"version": ".".join(h.groups()[:3]), "date": h.group(4),
                    "body": text[h.end(): nxt.start() if nxt else len(text)]})
    return out

Now the download. It needs the network, so it is skipped by default; the saved file ships with the course.

In [11]:
def get(url: str) -> str:
    return urllib.request.urlopen(url, timeout=20).read().decode()

fetched = {"fetched": str(date.today()), "packages": {}}
for pkg, url in WATCH.items():
    meta = json.loads(get(f"https://pypi.org/pypi/{pkg}/json"))
    first = {v: min(f["upload_time"][:10] for f in fs) for v, fs in meta["releases"].items() if fs}
    fetched["packages"][pkg] = {
        "latest": meta["info"]["version"], "latest_date": first[meta["info"]["version"]],
        "releases_last_year": sum(d >= str(AS_OF - timedelta(days=365)) for d in first.values()),
        "entries": entries_since(get(url), INSTALLED[pkg])}
Path("../data/recorded/59.json").write_text(json.dumps(fetched, indent=1))
print({p: (d["latest"], len(d["entries"])) for p, d in fetched["packages"].items()})

⏭  Skipped: this cell needs [download]. Run it yourself, or re-run with TUT_ENABLE=download.


### 7. The routine: triage
Read the saved fetch (recorded output, fetched on 2026-10-11, when this lesson was built). Triage each
changelog line: **skip** chores, docs and internal changes; **act** if it says breaking, deprecated,
removed, retired or renamed; **read** if it is a feature or touches a topic a lesson covers.

In [12]:
RECORDED = json.loads(Path("../data/recorded/59.json").read_text())
ACT = re.compile(r"breaking|deprecat|remov|retire|sunset|rename", re.I)
HINTS = {r"thinking|effort": "40", r"tool runner|AFC|toolset|mcp": "44", r"interaction|conversation": "09",
         r"structured|output_config": "20", r"cach": "41", r"batch": "42", r"httpx|retry|timeout": "14",
         r"claude-|gemini-|gpt-": "53"}

def triage(entry: dict) -> list[dict]:
    section, items = "", []
    for line in entry["body"].splitlines():
        if line.startswith("### "):
            section = line[4:].strip()
        elif line.startswith("* "):
            text = re.sub(r"\s*\(\[[^\]]+\]\([^)]+\)\)", "", line[2:]).strip()
            lessons = sorted({l for pat, l in HINTS.items() if re.search(pat, text, re.I)})
            if section in ("Chores", "Documentation") or text.startswith("**internal:**"):
                level = "skip"
            else:
                level = "act" if ACT.search(text) else "read" if section == "Features" or lessons else "skip"
            items.append({"level": level, "text": text, "lessons": lessons, "version": entry["version"]})
    return items

print(json.dumps(triage(RECORDED["packages"]["google-genai"]["entries"][0])[:3], indent=1)[:600])

[
 {
  "level": "read",
  "text": "Support `\"allowlist\": \"disabled\"` in environment network config.",
  "lessons": [],
  "version": "2.29.0"
 },
 {
  "level": "read",
  "text": "Update discovery doc",
  "lessons": [],
  "version": "2.29.0"
 },
 {
  "level": "read",
  "text": "Only log direct AFC warning when functions are present in the config",
  "lessons": [
   "44"
  ],
  "version": "2.29.0"
 }
]


### 8. The routine: one to-do list
One function puts it together: the changelog items worth a look, plus dated deadlines from a
**retirement calendar**. Anthropic gives Claude Haiku 4.5 a retirement "not sooner than" 2026-10-15
(RESEARCH.md), so a deadline 4 days after `AS_OF` must show up.

In [13]:
RETIREMENTS = {"claude-haiku-4-5": date(2026, 10, 15), "OpenAI Assistants API": date(2026, 8, 26)}

def weekly_check(recorded: dict, as_of: date, warn_days: int = 30) -> list[str]:
    todo = []
    for name, day in RETIREMENTS.items():
        days = (day - as_of).days
        if 0 <= days <= warn_days:
            todo.append(f"[deadline] {name}: {day} ({days} days) -> lesson 53")
        elif days < 0:
            todo.append(f"[passed]   {name}: {day}; check nothing still calls it")
    for pkg, d in recorded["packages"].items():
        items = [i for e in d["entries"] for i in triage(e)]
        todo.append(f"[{pkg}] {INSTALLED[pkg]} -> {d['latest']} ({d['latest_date']}): "
                    f"{sum(i['level'] == 'act' for i in items)} act, {sum(i['level'] == 'read' for i in items)} read, "
                    f"{sum(i['level'] == 'skip' for i in items)} skip")
        todo += [f"   {i['level']:4} {i['version']:7} {i['text'][:70]} -> {i['lessons'] or '-'}"
                 for i in items if i["level"] == "act" or (i["level"] == "read" and i["lessons"])]
    return todo

print("\n".join(weekly_check(RECORDED, AS_OF)))

[deadline] claude-haiku-4-5: 2026-10-15 (4 days) -> lesson 53
[passed]   OpenAI Assistants API: 2026-08-26; check nothing still calls it
[anthropic] 1.13.0 -> 1.13.0 (2026-10-09): 1 act, 9 read, 22 skip
   read 1.12.0  **api:** add claude-haiku-5-5 and typed computer and browser toolset t -> ['44', '53']
   read 1.12.0  **api:** add disabled to thinking types in model capabilities -> ['40']
   act  1.12.0  **api:** add display_name to RBAC roles and deprecate name -> -
[openai] 3.28.0 -> 3.28.0 (2026-10-09): 0 act, 8 read, 4 skip
[google-genai] 2.29.0 -> 2.29.0 (2026-10-07): 0 act, 3 read, 3 skip
   read 2.29.0  Only log direct AFC warning when functions are present in the config -> ['44']


Release pace from the same fetch: how many versions each SDK shipped in the year before `AS_OF`. This is
why the check is weekly and not "when something breaks".

In [14]:
for pkg, d in RECORDED["packages"].items():
    print(f"{pkg:13} {d['releases_last_year']:3} releases in 12 months (~{d['releases_last_year'] / 52:.1f} a week)")

anthropic      87 releases in 12 months (~1.7 a week)
openai         94 releases in 12 months (~1.8 a week)
google-genai   69 releases in 12 months (~1.3 a week)


## At real scale
- **The SDKs ship more than once a week.** The fetch above counts 69 to 94 releases a year per SDK. Teams
  pin exact versions, upgrade on a schedule (weekly or fortnightly), and run the recorded tests (lesson 47)
  and the golden-set eval (lesson 48) on every bump.
- **Preview APIs break.** google-genai 1.68.0 (2026-03-17) made two breaking changes to the Interactions
  API while it was in preview. Teams keep previews behind an adapter (lesson 37) until GA.
- **Server-side changes don't show in the SDK.** The installed openai SDK still has `beta.assistants`;
  the API behind it shut down on 2026-08-26. Only a dated calendar and a canary call (lesson 58) catch that.
- **Watch:** days to the nearest retirement, SDK versions behind latest, "act" items open.

## Decision guide
| Situation | Approach | Why |
|---|---|---|
| A new API is GA and your platform has it | Adopt behind your adapter; compare on the eval | Same loop, measured change |
| A new API is preview, or not on your platform | Watch; re-check on a date | Previews break; Vertex lacked Interactions |
| A retirement date is under 30 days away | Migrate now (lesson 53) | Requests start failing on the day |
| An SDK major version | Read MIGRATION.md, run recorded tests | Majors change transport and remove APIs |
| A row in your table has no probe | Find a probe, or mark it "docs only" with the date read | Notes drift; probes don't |

## What just happened
- Nine dated changes went into a table, each with its source and the lessons it touches.
- Probes ran against the installed SDKs: anthropic and openai sit on httpx2, but google-genai 2.28 still
  sits on httpx, so one row from the research notes was wrong and was corrected from the changelog.
- The effort dial has three spellings; Scout's `minimal` maps to `low` on Anthropic.
- Scout's parts got a verdict each: Interactions API is "watch" (404 on Vertex); cassette checks per HTTP
  library is "adopt now".
- A changelog routine fetched PyPI and changelogs once (recorded), triaged lines into act/read/skip and
  produced a to-do list with a 4-day deadline for Claude Haiku 4.5.

## Common mistakes
**1. Comparing versions as strings.** `"1.9.0"` sorts after `"1.13.0"` because `"9" > "1"`, so a routine
that does this decides you are already on the newest version.

In [15]:
newest = max(["1.9.0", "1.13.0", "1.12.1"])
assert newest == "1.13.0", f"string max picked {newest}"

AssertionError: string max picked 1.9.0

Fix: compare tuples of integers, as `as_tuple` does (or use `packaging.version.Version`).

In [16]:
print(max(["1.9.0", "1.13.0", "1.12.1"], key=as_tuple))

1.13.0


**2. Writing a row from memory or notes, without a probe.** The research notes said all three SDKs moved
to httpx2. A test suite that patches only httpx2 would then silently miss every Gemini call.

In [17]:
libs = PROBES["httpx2"]()
assert all(lib == "httpx2" for lib in libs.values()), f"not all on httpx2: {libs}"

AssertionError: not all on httpx2: {'anthropic': 'httpx2', 'openai': 'httpx2', 'google-genai': 'httpx'}

Fix: store what the probe says, and let the check fail loudly when an upgrade changes it.

In [18]:
EXPECTED_HTTP = {"anthropic": "httpx2", "openai": "httpx2", "google-genai": "httpx"}   # probed on AS_OF
print("matches the probe:", PROBES["httpx2"]() == EXPECTED_HTTP)

matches the probe: True


## Try it
Add a "how far behind" line to the routine. Write `behind(installed, latest)` that returns
`(majors, minors)` behind (minors counted only within the same major), and add a `[upgrade]` to-do for
any package behind by a major or by 2 or more minors. Run it on `RECORDED` and `INSTALLED`.

In [19]:
# your code here

## Key terms
- **frontier table** — dated recent vendor changes, each with source, affected lessons and a probe.
- **probe** — an offline check that a claim about an SDK holds in your installed version.
- **Interactions API** — Gemini's recommended API since June 2026, with server-side state.
- **effort controls** — how much a model thinks: `thinking_level`, `output_config.effort`, `reasoning.effort`.
- **server-side tools** — tools the vendor runs in your request, pinned by a dated type.
- **context engineering** — choosing the smallest set of tokens each model call needs.
- **changelog triage** — sorting changelog lines into act, read or skip.
- **retirement calendar** — dated model and API shutdowns you depend on, checked on a schedule.